# DATA 37711 · Homework 3


All of the code can be found at:
https://github.com/ryanmichauduc/DATA_37711_AUT27


## 1. Boosting for count data
**(a)** A Poisson distribution assigns mass to nonnegative integer counts. It is a reasonable starting model when the conditional variance is comparable to the conditional mean. A tree ensemble can output any real score. Setting $\mu(x)=e^{f(x)}$ guarantees a positive rate and allows the ensemble to add effects on the log-mean scale. Technically, the log is the link and the exponential is its inverse.

**(b)** Taking minus the logarithm of $P(Y=y\mid x)=e^{-\mu}\mu^y/y!$ gives the per-observation loss
$$L(y,f)=e^{f(x)}-yf(x)+\log(y!).$$
Average this over the training data. The factorial term is constant during optimization.

**(c)** Differentiating with respect to the score gives $\partial L/\partial f(x)=e^{f(x)}-y$.

**(d)** For a positive sample mean, initialize $f_0=\log\bar y$. At stage $j$, compute $r_i=y_i-e^{f_{j-1}(x_i)}$ and fit a regression tree $h_j$ by least squares to these negative gradients. Choose a step size by minimizing the training Poisson loss along $f_{j-1}+\lambda h_j$, optionally shrinking the step. Add the scaled tree and recompute the residuals.

## 2. Hyperparameter selection
**(a)** Let $k$ count candidate configurations. Condition on the training set and fitting randomness so that all $k$ fitted predictors are fixed independently of the $l$ i.i.d. test examples. Assume loss in $[0,B]$. For any fixed candidate $j$, Hoeffding's inequality gives
$$P(|\hat R_j-R_j|>t)\le 2\exp(-2lt^2/B^2).$$
A union bound multiplies the right side by $k$. Setting it equal to $\delta$ shows that with probability at least $1-\delta$, the test-selected minimizer $\hat j$ satisfies
$$|\hat R_{\hat j}-R_{\hat j}|\le B\sqrt{\frac{\log(2k/\delta)}{2l}}.$$
The simultaneous event covers every possible selected index. Removing the conditioning preserves the probability bound. Training size $n$ may affect true risk but does not enter this holdout estimate. Unrestricted squared loss needs additional tail assumptions. For the wine problem, predictions from both averaging models stay within the training target range, so scores and predictions in $[1,10]$ give $B=81$.

**(b)** Selecting the smallest observed error favors models that benefited from sampling noise. In expectation, the minimum estimated risk is at most the minimum true risk, which is itself at most the expected true risk of the selected model. The bound weakens as $\log k/l$ grows. A fresh test set gives an independent estimate after selection.

**(c)** The code uses a single random 70/10/20 split. KNN varies neighborhood size, uniform versus inverse-distance weighting, and Manhattan versus Euclidean distance after scaling the features. The forest varies depth, minimum leaf size, features considered per split and tree count. Each class gets more configurations than validation observations. Both winners are chosen before scoring either on the test set.

[Scikit-learn KNeighborsRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) · [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_squared_error, log_loss

def locate(name):
    for folder in (Path.cwd(), Path.cwd() / 'upload'):
        if (folder / name).is_file():
            return folder / name
    return None

In [2]:
def wine_search(path):
    wine = pd.read_csv(path, sep=None, engine='python')
    X, y = wine.drop(columns='quality'), wine['quality']
    X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.125, random_state=42)
    print('Train, validation, test sizes:', len(y_train), len(y_val), len(y_test))
    scaler = StandardScaler().fit(X_train)
    knn_grid = dict(n_neighbors=range(1, len(y_val)//4 + 2), weights=['uniform','distance'], p=[1,2])
    forest_grid = dict(max_depth=[2,4,6,8,12,None], min_samples_leaf=[1,2,4,8],
                       max_features=[0.4,0.7,1.0],
                       n_estimators=range(10, 10*(len(y_val)//72+2), 10))
    searches = [('KNN', KNeighborsRegressor(), knn_grid,
                 scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)),
                ('Forest', RandomForestRegressor(random_state=42,n_jobs=-1), forest_grid,
                 X_train, X_val, X_test)]
    winners = []
    for name, model, params, train, val, test in searches:
        grid = ParameterGrid(params)
        assert len(grid) > len(y_val)
        best = np.inf
        for config in grid:
            model.set_params(**config).fit(train, y_train)
            score = mean_squared_error(y_val, model.predict(val))
            if score < best:
                best, best_config = score, config
        model.set_params(**best_config).fit(train, y_train)
        winners.append((name, model, test, best, best_config, len(grid)))
    for name, model, test, val_mse, config, size in winners:
        test_mse = mean_squared_error(y_test, model.predict(test))
        print(f'{name} ({size} configurations): {config}')
        print(f'Validation MSE {val_mse:.4f}, test MSE {test_mse:.4f}. '
              f'Test minus validation = {test_mse-val_mse:+.4f}.')

wine_path = locate('winequality-white.csv')
if wine_path is None:
    print('Add winequality-white.csv beside this notebook and rerun this cell for the MSE comparison.')
else:
    wine_search(wine_path)

Add winequality-white.csv beside this notebook and rerun this cell for the MSE comparison.


A positive test-minus-validation gap means the selected configuration performed better on validation. Selection tends to favor low validation MSE, but sampling variation allows either sign on one split.

## 3. Generalization as trees are added
Both datasets have a binary first column. For each, hold out a stratified 25% sample and fit depth-two trees with learning rate 0.1. Staged predictions evaluate the exact requested ensemble sizes from one fit. Classification error measures incorrect labels, while log loss also penalizes confident mistakes.

In [3]:
def boosting_results(filename):
    data = np.loadtxt(locate(filename), delimiter=',')
    X, y = data[:, 1:], data[:, 0]
    train, test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=42)
    model = GradientBoostingClassifier(n_estimators=1000, max_depth=2,
                                       learning_rate=0.1, random_state=42).fit(train, y_train)
    rows = []
    for stage, (p, q) in enumerate(zip(model.staged_predict_proba(train),
                                       model.staged_predict_proba(test)), start=1):
        if stage in (1, 10, 100, 300, 1000):
            rows.append([stage, (p.argmax(axis=1) != y_train).mean(),
                         (q.argmax(axis=1) != y_test).mean(), log_loss(y_train,p), log_loss(y_test,q)])
    return pd.DataFrame(rows, columns=['trees','train_error','test_error','train_log_loss','test_log_loss'])

for filename in ('dset0-1.csv', 'dset1-1.csv'):
    print(filename)
    print(boosting_results(filename).round(4).to_string(index=False))

dset0-1.csv
 trees  train_error  test_error  train_log_loss  test_log_loss
     1       0.4267       0.424          0.6686         0.6716
    10       0.2533       0.320          0.5880         0.6097
   100       0.0373       0.216          0.2924         0.4457
   300       0.0053       0.208          0.1245         0.4604
  1000       0.0000       0.176          0.0137         0.7104
dset1-1.csv
 trees  train_error  test_error  train_log_loss  test_log_loss
     1       0.4533       0.456          0.6441         0.6605
    10       0.1547       0.272          0.4508         0.5394
   100       0.0427       0.224          0.1931         0.3951
   300       0.0080       0.240          0.0802         0.4667
  1000       0.0000       0.248          0.0081         0.7472


**(a) dset0.** Held-out classification error falls from 42.4% with one tree to 17.6% with 1,000 trees. Held-out log loss reaches its lowest measured value at 100 trees (0.446) and rises to 0.710. The final model labels more examples correctly, but the probabilities it assigns to the observed outcomes are worse on average under log loss.

**(b) dset1.** Classification error falls from 45.6% to 22.4% at 100 trees, then rises to 24.8% at 1,000 trees. Log loss also reaches its lowest measured value at 100 trees. Additional rounds improve training fit while held-out performance deteriorates.

**(c)** Class overlap or label noise could make late trees more likely to fit idiosyncratic examples in dset1. Another possibility is that dset0 has boundary structure that takes more trees to learn. The difference between 100 and 1,000 trees amounts to five fewer mistakes on dset0 and three more on dset1 out of 125 held-out examples each. One split cannot establish which explanation generated the data.